# Capstone II: Revisiting Monte Carlo Estimation of Pi

> **Prerequisites:** This capstone assumes you've read **Chapters 2–9**.

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
# we need the notebook-utils and openai extras for this notebook
uv sync --extra notebook-utils --extra openai

# to launch the notebook
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPI by uncommenting the cell below.

<!-- provider-note:start -->
## Setting up the LLM

I recommend running these notebooks on **Ollama Cloud**. It serves much
larger models than most laptops can run. The default cloud model is picked
by benchmarking candidates on these notebooks, and an automated run checks
every notebook against it each week. The
[notebook status page](https://masfromscratch.com/notebook-status/) shows
the current model and results.

1. Create an API key at https://ollama.com/settings/keys.
2. Export it before launching Jupyter: `export OLLAMA_API_KEY=<your key>`.
3. That's all. When a notebook builds its LLM, it detects the key and uses
   Ollama Cloud.

**Prefer to run locally?** Install Ollama from https://ollama.com/download
and pull the model with `ollama pull qwen3:14b`. If no key is set, the
notebook starts the Ollama service for you, so there is no need to run
`ollama serve`. Local models are smaller, so your outputs will differ from
the recorded ones.

**Prefer OpenAI or Anthropic?**

1. Install the extra: `uv sync --extra openai` or `uv sync --extra anthropic`.
2. Export `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` before launching Jupyter.
   If you forget, you will be prompted for it, which keeps the key out of
   the saved notebook.
3. Set `LLM_PROVIDER=openai` (or `anthropic`), or pass `provider="openai"`
   to `make_llm()`. A key on its own never switches providers, so one
   exported for unrelated work cannot reroute you.

The switch applies wherever a notebook builds its LLM with `make_llm()`.
A notebook that constructs `OllamaLLM` directly stays on Ollama. Output on
gpt-5 or Claude will differ from the recorded outputs, and prompt-sensitive
examples (the ch09 evaluator pattern, ch08 supervised trajectories) may
behave noticeably differently.
<!-- provider-note:end -->

In [1]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install 'llm-agents-from-scratch[notebook-utils,openai]'

## Setup

In [2]:
import logging
import os

from llm_agents_from_scratch.logger import enable_console_logging

### Constants

In [3]:
IS_ON_RUNPOD = "RUNPOD_POD_ID" in os.environ
LOGGING_ENABLED = False  # 20 concurrent loops make for very long logs
LOGGING_LEVEL = logging.INFO

# the experiment
NUM_ESTIMATES = 20  # K independent estimates of pi per run
MAX_STEPS_PER_ESTIMATE = 20  # Capstone I's budget for a single estimate
MAX_DISPATCHES_PER_STEP = 5  # stays under the provider's concurrency cap
NUM_REPLICATIONS = 3

In [4]:
# Install additional dependencies for notebook
if IS_ON_RUNPOD:
    !uv pip install numpy pandas --system
else:
    !uv pip install numpy pandas

Using Python 3.13.5 environment at: /home/nerdai/Projects/llm-agents-from-scratch/.venv
Audited 2 packages in 1ms


In [5]:
# maybe enable logging
if LOGGING_ENABLED:
    enable_console_logging(LOGGING_LEVEL)

## The Task: the standard error of Capstone I's algorithm

Capstone I's agent produced a point estimate of pi. This capstone asks how
much those estimates vary: what is the standard error of Capstone I's
algorithm? To answer it, we run the whole algorithm `NUM_ESTIMATES` times,
each time on its own new sample, and compute

$$\text{SE} = \frac{s}{\sqrt{K}}$$

where $s$ is the standard deviation of the $K$ final estimates.

A single agent doing this must track $K$ samples, $K$ doubling schedules
and $K$ convergence states in one growing rollout. That is the context
sprawl that Chapter 9 sets out to tame with subagents, so we build both
systems and compare them.

## LLMs

In [6]:
from llm_agents_from_scratch.notebook_utils import make_llm

### (Listing 1) Measuring the context sent to an LLM

To compare how much context each system sends, we wrap the LLM in a
`BaseLLM` that forwards every call and counts the prompt characters it
sends: the prompt itself, the chat history and any tool results. Because
the wrapper is itself a `BaseLLM`, agents and subagents use it like any
other LLM.

In [7]:
from collections.abc import Sequence
from typing import Any

from llm_agents_from_scratch.base.llm import BaseLLM
from llm_agents_from_scratch.data_structures import ChatMessage


def history_chars(chat_history: Sequence[ChatMessage]) -> int:
    """Total characters in a chat history."""
    return sum(len(m.content) for m in chat_history)


class MeteredLLM(BaseLLM):
    """An LLM wrapper that counts the prompt characters sent to the LLM.

    Attributes:
        llm (BaseLLM): The wrapped LLM.
        prompt_chars (int): Prompt characters sent since the last reset.
    """

    def __init__(self, llm: BaseLLM) -> None:
        """Initialize a MeteredLLM.

        Args:
            llm (BaseLLM): The LLM to wrap.
        """
        self.llm = llm
        self.prompt_chars = 0

    def reset(self) -> None:
        """Reset the prompt character count."""
        self.prompt_chars = 0

    async def complete(self, prompt: str, **kwargs: Any) -> Any:
        """Count, then forward to the wrapped LLM."""
        self.prompt_chars += len(prompt)
        return await self.llm.complete(prompt, **kwargs)

    async def structured_output(
        self,
        prompt: str,
        mdl: Any,
        **kwargs: Any,
    ) -> Any:
        """Count, then forward to the wrapped LLM."""
        self.prompt_chars += len(prompt)
        return await self.llm.structured_output(prompt, mdl, **kwargs)

    async def chat(
        self,
        input: str,
        chat_history: Sequence[ChatMessage] | None = None,
        tools: Sequence[Any] | None = None,
        **kwargs: Any,
    ) -> Any:
        """Count, then forward to the wrapped LLM."""
        self.prompt_chars += len(input) + history_chars(chat_history or [])
        return await self.llm.chat(
            input=input,
            chat_history=chat_history,
            tools=tools,
            **kwargs,
        )

    async def continue_chat_with_tool_results(
        self,
        tool_call_results: Sequence[Any],
        chat_history: Sequence[ChatMessage],
        tools: Sequence[Any] | None = None,
        **kwargs: Any,
    ) -> Any:
        """Count, then forward to the wrapped LLM."""
        self.prompt_chars += history_chars(chat_history) + sum(
            len(str(r.content)) for r in tool_call_results
        )
        return await self.llm.continue_chat_with_tool_results(
            tool_call_results=tool_call_results,
            chat_history=chat_history,
            tools=tools,
            **kwargs,
        )

## Build Tools

### (Listing 2) The Capstone I tools, with one change

The three tools are Capstone I's, with one change: `monte_carlo_estimate`
now also reports whether the estimate is in the target range
`[3.1415, 3.1425)`. In Capstone I the agent made that comparison itself,
and LLMs occasionally get it wrong (declaring 3.1436 "in range", say). That
mistake has nothing to do with context, so here the tool makes the check
and both systems read its answer.

In [8]:
import uuid

import numpy as np
from pydantic import BaseModel, ConfigDict, Field

from llm_agents_from_scratch.tools import PydanticFunctionTool

# sample_id -> (points inside, total points), as in Capstone I
SAMPLE_REGISTRY: dict[str, tuple[int, int]] = {}

# points are drawn this many at a time, so memory stays small for any n
CHUNK_SIZE = 1_000_000

TARGET_LOWER, TARGET_UPPER = 3.1415, 3.1425


def count_inside(n: int) -> int:
    """Draw n random points; return how many land inside the circle."""
    inside = 0
    for start in range(0, n, CHUNK_SIZE):
        size = min(CHUNK_SIZE, n - start)
        x, y = np.random.uniform(size=(2, size))
        inside += int(np.count_nonzero(x**2 + y**2 < 1))
    return inside


def estimate_of(sample_id: str) -> float:
    """The Monte Carlo estimate of pi from a registered sample."""
    inside, n = SAMPLE_REGISTRY[sample_id]
    return (inside / n) * 4


class RandomSampleParams(BaseModel):
    """Params for generate_random_sample."""

    model_config = ConfigDict(extra="forbid")
    n: int = Field(description="The number of random points to generate")


class RandomSample(BaseModel):
    """Result from generate_random_sample and add_more_points_to_sample."""

    sample_id: str = Field(
        description="Pass this sample_id to monte_carlo_estimate",
    )
    sample_size: int

    def __str__(self) -> str:
        """String representation of RandomSample."""
        return self.model_dump_json()


def generate_random_sample(params: RandomSampleParams) -> RandomSample:
    """Generate n random points in [0, 1] × [0, 1].

    Returns a sample_id. Pass this sample_id directly to monte_carlo_estimate.
    """
    sample_id = str(uuid.uuid4())
    SAMPLE_REGISTRY[sample_id] = (count_inside(params.n), params.n)
    return RandomSample(sample_id=sample_id, sample_size=params.n)


class AddPointsParams(BaseModel):
    """Params for add_more_points_to_sample."""

    model_config = ConfigDict(extra="forbid")
    sample_id: str = Field(
        description="The sample_id of the sample to augment",
    )
    n: int = Field(description="The number of random points to generate")


def add_more_points_to_sample(params: AddPointsParams) -> RandomSample:
    """Add n more random points to an existing random sample.

    Returns a sample_id and the total number of points.
    """
    inside, total = SAMPLE_REGISTRY[params.sample_id]
    SAMPLE_REGISTRY[params.sample_id] = (
        inside + count_inside(params.n),
        total + params.n,
    )
    return RandomSample(
        sample_id=params.sample_id,
        sample_size=total + params.n,
    )


class MonteCarloEstimateParams(BaseModel):
    """Params for monte_carlo_estimate."""

    model_config = ConfigDict(extra="forbid")
    sample_id: str = Field(
        description="The sample_id returned by generate_random_sample",
    )


class MonteCarloEstimateResult(BaseModel):
    """Results for monte_carlo_estimate."""

    sample_id: str
    sample_size: int
    estimate: float
    in_target_range: bool

    def __str__(self) -> str:
        """String representation of MonteCarloEstimateResult."""
        return self.model_dump_json()


def monte_carlo_estimate(
    params: MonteCarloEstimateParams,
) -> MonteCarloEstimateResult:
    """Estimate pi using Monte Carlo method.

    Also reports whether the estimate is in the target range
    [3.1415, 3.1425).
    """
    estimate = estimate_of(params.sample_id)
    return MonteCarloEstimateResult(
        sample_id=params.sample_id,
        sample_size=SAMPLE_REGISTRY[params.sample_id][1],
        estimate=estimate,
        in_target_range=TARGET_LOWER <= estimate < TARGET_UPPER,
    )


random_sample_tool = PydanticFunctionTool(generate_random_sample)
add_more_points_tool = PydanticFunctionTool(add_more_points_to_sample)
monte_carlo_estimate_tool = PydanticFunctionTool(monte_carlo_estimate)

### (Listing 3) Tool: `compute_standard_error()`

In [9]:
import statistics


class StandardErrorParams(BaseModel):
    """Params for compute_standard_error."""

    model_config = ConfigDict(extra="forbid")
    sample_ids: list[str] = Field(
        description="The sample_ids of the independent estimates of pi",
    )


class StandardErrorResult(BaseModel):
    """Result of compute_standard_error."""

    num_estimates: int
    mean: float
    standard_error: float

    def __str__(self) -> str:
        """String representation of StandardErrorResult."""
        return self.model_dump_json()


def compute_standard_error(params: StandardErrorParams) -> StandardErrorResult:
    """Compute the mean and standard error of independent estimates of pi."""
    estimates = [estimate_of(s) for s in params.sample_ids]
    k = len(estimates)
    return StandardErrorResult(
        num_estimates=k,
        mean=statistics.fmean(estimates),
        standard_error=statistics.stdev(estimates) / k**0.5 if k > 1 else 0.0,
    )


standard_error_tool = PydanticFunctionTool(compute_standard_error)

## System 1: A Single Agent

### (Listing 4) Writing the single agent's instruction

The algorithm, rules and output format follow Capstone I's instruction.
The agent now has to run the algorithm `NUM_ESTIMATES` times and finish by
computing the standard error.

In [10]:
algorithm = """
<algorithm>
1. Call generate_random_sample(1000000) to start a NEW sample of 1M points
2. Call monte_carlo_estimate(sample_id) to get the estimate
3. Check in_target_range in the result (True means the estimate is in
   [3.1415, 3.1425)):
   - True → this estimate is done
   - False → Continue to step 4
4. Call add_more_points_to_sample on the SAME sample, doubling the points
   each time:
   - First add: 1 million
   - Second add: 2 million
   - Third add: 4 million
   - And so on, doubling each iteration
5. After adding points, go back to step 2
</algorithm>
""".strip()

critical_rules = """
<critical_rules>
- If the task is not complete, your response MUST contain a tool call
- Do not just describe what you plan to do—actually call the tool
- NEVER fabricate tool results-only use actual tool responses
- NEVER invent a sample_id
</critical_rules>
""".strip()

final_output = """
<final_output>
When you have the standard error, respond with this exact JSON structure
and nothing else:

{"sample_ids": [<the sample_ids>], "standard_error": <the standard error>}

No explanation, no markdown formatting, no code blocks—just the raw JSON.
</final_output>
""".strip()

single_agent_instruction = f"""
You are tasked with estimating the standard error of a Monte Carlo
algorithm for estimating pi. Run the algorithm below {NUM_ESTIMATES} times,
each time with its own NEW sample, so you end up with {NUM_ESTIMATES}
independent estimates that are each in the target range. Then call
compute_standard_error with the {NUM_ESTIMATES} sample_ids.

{algorithm}

{critical_rules}

{final_output}
""".strip()

### (Listing 5) Creating the single agent

In [11]:
from llm_agents_from_scratch import LLMAgent

single_agent_llm = MeteredLLM(make_llm())
single_agent = LLMAgent(
    llm=single_agent_llm,
    tools=[
        random_sample_tool,
        add_more_points_tool,
        monte_carlo_estimate_tool,
        standard_error_tool,
    ],
)
single_agent_max_steps = MAX_STEPS_PER_ESTIMATE * NUM_ESTIMATES

✓ Using Ollama Cloud (deepseek-v4.1-flash)


## System 2: A Coordinator and Subagents

### (Listing 6) The `pi-estimator` subagent

Each `pi-estimator` dispatch runs Capstone I's algorithm once, on its own
new sample, in a fresh context. The algorithm goes in the subagent's
system message, by overriding the `system_message` template, so the
coordinator's dispatch can be as short as "Estimate pi."

In [12]:
from llm_agents_from_scratch import LLMAgentBuilder
from llm_agents_from_scratch.agent.templates import default_templates
from llm_agents_from_scratch.subagents import SubAgentSpec

subagents_llm = MeteredLLM(make_llm())

pi_estimator_system_message = (
    default_templates["system_message"]
    + "\nWhen asked to estimate pi, follow this algorithm until "
    "in_target_range is True, then report the sample_id and the "
    "estimate.\n\n" + algorithm + "\n\n" + critical_rules
)

pi_estimator_spec = SubAgentSpec(
    name="pi-estimator",
    description=(
        "Estimates pi with its own new Monte Carlo sample until the "
        "estimate is in the target range [3.1415, 3.1425). Returns the "
        "sample_id and the estimate."
    ),
    builder=LLMAgentBuilder(
        llm=subagents_llm,
        tools=[
            random_sample_tool,
            add_more_points_tool,
            monte_carlo_estimate_tool,
        ],
        templates={
            **default_templates,
            "system_message": pi_estimator_system_message,
        },
    ),
    max_steps=MAX_STEPS_PER_ESTIMATE,
)

✓ Using Ollama Cloud (deepseek-v4.1-flash)


### (Listing 7) The coordinator

The coordinator holds only `compute_standard_error`. It dispatches the
estimates in batches, since dispatches in the same response run
concurrently and a cloud provider caps how many requests it serves at
once.

In [13]:
coordinator_instruction = f"""
You are tasked with estimating the standard error of a Monte Carlo
algorithm for estimating pi. You need {NUM_ESTIMATES} independent
estimates of pi.

1. Dispatch the pi-estimator subagent {NUM_ESTIMATES} times in total, in
   batches of at most {MAX_DISPATCHES_PER_STEP} dispatches per response
   (dispatches in the same response run concurrently). Each dispatch's
   task is simply: "Estimate pi."
2. Collect the {NUM_ESTIMATES} sample_ids from the results.
3. Call compute_standard_error with the {NUM_ESTIMATES} sample_ids.

{critical_rules}

{final_output}
""".strip()

coordinator = LLMAgent(
    llm=subagents_llm,
    tools=[standard_error_tool],
    subagents=[pi_estimator_spec],
)
coordinator_max_steps = MAX_STEPS_PER_ESTIMATE

## Evaluation

### (Listing 8) Evaluating Task Success

Success is checked against ground truth, the sample registry. The final
response must name exactly `NUM_ESTIMATES` distinct, real samples, every
one of them in the target range, and report the standard error that those
samples give, to within 1% (enough for an agent that rounds the tool's
value). Unlike Capstone I, we pick the JSON out of the final response
instead of requiring it verbatim. Chapter 4's `LLMAgent` returns its
deciding step's response as the final answer, which often adds a sentence
before the JSON, and this capstone is about context, not formatting.

In [14]:
import json
import math


def final_json(content: str) -> dict | None:
    """The last JSON object in `content` that has a `sample_ids` key."""
    decoder = json.JSONDecoder()
    found = None
    for i, char in enumerate(content):
        if char != "{":
            continue
        try:
            obj, _ = decoder.raw_decode(content, i)
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict) and "sample_ids" in obj:
            found = obj
    return found


def is_task_success(
    handler: LLMAgent.TaskHandler,
    verbose: bool = False,
) -> bool:
    """Determines task success against the sample registry.

    Args:
        handler (LLMAgent.TaskHandler): The handler containing the
            result or exception of the task execution
        verbose (bool): Whether to print out details of the
            determination. Defaults to False.

    Returns:
        bool: True if task was successful. False, otherwise.
    """
    if handler.exception():
        if verbose:
            print(handler.exception())
        return False

    output = final_json(handler.result().content)
    sample_ids = output.get("sample_ids") if output else None
    reported_se = output.get("standard_error") if output else None
    if (
        not isinstance(sample_ids, list)
        or len(sample_ids) != NUM_ESTIMATES
        or not all(isinstance(s, str) for s in sample_ids)
        or len(set(sample_ids)) != NUM_ESTIMATES
    ):
        if verbose:
            print(f"Expected {NUM_ESTIMATES} distinct sample_ids: {output}")
        return False

    real = [s for s in sample_ids if s in SAMPLE_REGISTRY]
    in_range = [
        s for s in real if TARGET_LOWER <= estimate_of(s) < TARGET_UPPER
    ]
    se_ok = False
    if len(real) == NUM_ESTIMATES and isinstance(reported_se, int | float):
        params = StandardErrorParams(sample_ids=sample_ids)
        true_se = compute_standard_error(params).standard_error
        se_ok = math.isclose(reported_se, true_se, rel_tol=0.01)
    if verbose:
        print(
            f"{len(real)} real, {len(in_range)} in range, "
            f"standard_error {reported_se} correct: {se_ok}",
        )
    return len(in_range) == NUM_ESTIMATES and se_ok

### (Listing 9) Running a task and measuring its context

Each run records task success alongside four measures of the context it
took: wall time, the number of steps and the size of the agent's own
rollout, and the prompt characters sent to the LLM. For the coordinator,
the prompt characters include every subagent's calls, since they share
the coordinator's `MeteredLLM`. Divide by about four for a rough count of
tokens.

In [15]:
import contextlib
import time

from llm_agents_from_scratch.data_structures import Task


async def run_and_measure(
    agent: LLMAgent,
    metered_llm: MeteredLLM,
    instruction: str,
    max_steps: int,
) -> tuple[LLMAgent.TaskHandler, dict[str, Any]]:
    """Run a task to completion and measure the context it took."""
    metered_llm.reset()
    start = time.perf_counter()
    handler = agent.run(Task(instruction=instruction), max_steps=max_steps)
    # a failed run is recorded through handler.exception()
    with contextlib.suppress(Exception):
        await handler
    return handler, {
        "task_success": is_task_success(handler),
        "wall_seconds": round(time.perf_counter() - start),
        "steps": handler.rollout.count("=== Task Step Start ==="),
        "rollout_chars": len(handler.rollout),
        "prompt_chars": metered_llm.prompt_chars,
    }

## Perform the Task

In [16]:
single_handler, single_metrics = await run_and_measure(
    single_agent,
    single_agent_llm,
    single_agent_instruction,
    single_agent_max_steps,
)
single_metrics

{'task_success': True,
 'wall_seconds': 111,
 'steps': 23,
 'rollout_chars': 141580,
 'prompt_chars': 6420526}

In [17]:
is_task_success(single_handler, verbose=True)

20 real, 20 in range, standard_error 5.7676716588324453e-05 correct: True


True

In [18]:
coordinator_handler, coordinator_metrics = await run_and_measure(
    coordinator,
    subagents_llm,
    coordinator_instruction,
    coordinator_max_steps,
)
coordinator_metrics

{'task_success': True,
 'wall_seconds': 131,
 'steps': 5,
 'rollout_chars': 24079,
 'prompt_chars': 1926524}

In [19]:
is_task_success(coordinator_handler, verbose=True)

20 real, 20 in range, standard_error 5.9720974768524894e-05 correct: True


True

The coordinator's rollout shows the whole run: each batch of dispatches,
the `pi-estimator` results, and the final `compute_standard_error` call.
The single agent's rollout is several times longer, so we print only its
final answer.

In [20]:
print(coordinator_handler.rollout)

=== Task Step Start ===

💬 assistant: My current instruction is 'You are tasked with estimating the standard error of a Monte Carlo
algorithm for estimating pi. You need 20 independent
estimates of pi.

1. Dispatch the pi-estimator subagent 20 times in total, in
   batches of at most 5 dispatches per response
   (dispatches in the same response run concurrently). Each dispatch's
   task is simply: "Estimate pi."
2. Collect the 20 sample_ids from the results.
3. Call compute_standard_error with the 20 sample_ids.

<critical_rules>
- If the task is not complete, your response MUST contain a tool call
- Do not just describe what you plan to do—actually call the tool
- NEVER fabricate tool results-only use actual tool responses
- NEVER invent a sample_id
</critical_rules>

<final_output>
When you have the standard error, respond with this exact JSON structure
and nothing else:

{"sample_ids": [<the sample_ids>], "standard_error": <the standard error>}

No explanation, no markdown formattin

In [21]:
print(single_handler.result().content)

{"sample_ids": ["ca2044f9-70f0-49fe-9547-7237a2a22f43", "6592ba69-40ea-43c8-9d0d-548c55f637a3", "91b6c2fb-4670-40c2-a11e-87472024857c", "17683f3c-132f-4016-8187-001420b942c8", "d81f4ebe-5a99-4d2a-9424-933d736426d0", "32997b61-5a7c-48f5-9a4f-5aa537088cc0", "4176fef0-4429-4a98-9f01-aaad00a16bcf", "ead003b8-764e-4c17-a3d8-9f1ce07b82df", "cd4dae11-11ff-4954-b6d2-6fd06c010c4b", "1b7a9063-7a5b-4225-bd5c-b4f624608703", "fd0692a5-4a51-430a-8772-46252bdf1064", "543f77a8-51ce-4765-8180-25ec8b5b1ccb", "7280069a-9c14-46ea-85e3-2054a9b0044a", "f12f7c2c-f3dd-40b3-b38d-b9a8610222eb", "d4812711-9a80-4379-90ac-940d660aee3f", "2176d480-0b48-4a36-9f78-5274070b57fd", "0c62be40-1b96-4332-b159-0118b7cef6e4", "ea0311a1-627d-431e-b9e8-07266fd61d22", "f216acbb-aece-4dec-b5fb-5dbcb729c944", "3fa75f25-f37f-480e-b39a-ad0e45f11133"], "standard_error": 0.000057676716588324453}


### Trajectory Analysis

As in Capstone I, an LLM judge reads each trajectory, here for the
failures that the ground-truth check cannot see.

In [22]:
trajectory_judge = make_llm(role="judge")

✓ Using Ollama Cloud (deepseek-v4.1-flash)


### (Listing 10) Rubric for LLM judge

In [23]:
class TrajectoryEvalRubric(BaseModel):
    """Rubric for evaluating an execution trajectory."""

    no_fabricated_results: bool = Field(
        description=(
            "True if every estimate, sample_id and standard error the agent "
            "used came from an actual tool or subagent result"
        ),
    )

    followed_doubling_schedule: bool = Field(
        description=(
            "True if every sample that needed more points was grown by "
            "doubling the points added each time. For a coordinator, judge "
            "from what the subagent results report."
        ),
    )

    followed_output_format: bool = Field(
        description=(
            "True if agent's final response contained only the JSON, "
            "with no additional text or explanation."
        ),
    )

    summary: str = Field(
        description="One sentence summary of trajectory quality",
    )

### (Listing 11) LLM judge instruction prompt

The prompt adds one rule to Capstone I's. A step often ends by announcing
its next tool call, which the following step then makes. Without the rule,
a judge counts an announced call and the call itself as two calls.

In [24]:
judge_prompt_template = """Evaluate this trajectory of an agent estimating
the standard error of a Monte Carlo algorithm for estimating pi.

The task: produce {num_estimates} independent estimates of pi, each on its
own new sample and each in the range [3.1415, 3.1425), growing a sample by
doubling the points added each time (1M, 2M, 4M, ...), then compute the
standard error of the estimates with compute_standard_error.

The agent is one of two kinds:
- a single agent that calls generate_random_sample,
  add_more_points_to_sample, monte_carlo_estimate and
  compute_standard_error itself
- a coordinator that dispatches pi-estimator subagents with
  `from_scratch__use_subagent` (each result is that subagent's final
  answer; its own tool calls are not part of the trajectory) and calls
  compute_standard_error itself

COUNTING TOOL CALLS: a step often ends with "I need to make the following
tool call(s)", announcing a call that the next step then makes. Count only
calls that are followed by a "🔧 tool:" result.

Note: If final_response is "Max steps error", the agent failed to complete
the task within the allowed number of steps.

<final_response>
{result}
</final_response>

<trajectory>
{trajectory}
</trajectory>

Evaluate and submit your judgment.""".strip()

In [25]:
async def judge(handler: LLMAgent.TaskHandler) -> TrajectoryEvalRubric:
    """Have the judge LLM evaluate a task's trajectory."""
    return await trajectory_judge.structured_output(
        prompt=judge_prompt_template.format(
            num_estimates=NUM_ESTIMATES,
            result=str(handler.exception() or handler.result()),
            trajectory=handler.rollout,
        ),
        mdl=TrajectoryEvalRubric,
    )


single_eval = await judge(single_handler)
print(single_eval.model_dump_json(indent=4))

{
    "no_fabricated_results": true,
    "followed_doubling_schedule": false,
    "followed_output_format": true,
    "summary": "The agent eventually obtained 20 in-range samples and a correct standard error, but it repeatedly violated the doubling schedule by prematurely re-adding points before verifying in-range status, causing many redundant and out-of-schedule additions."
}


In [26]:
coordinator_eval = await judge(coordinator_handler)
print(coordinator_eval.model_dump_json(indent=4))

{
    "no_fabricated_results": true,
    "followed_doubling_schedule": true,
    "followed_output_format": true,
    "summary": "The coordinator dispatched 20 pi-estimator subagents in four batches of 5, each subagent reported doubling sample sizes until in range, and the agent correctly computed the standard error and returned the exact JSON output."
}


## Replications for a more reliable evaluation

### (Listing 12) Repeated task executions with both systems

Runs go one after another rather than concurrently: each run already
makes many concurrent requests, and running several at once would hit the
provider's concurrency cap.

In [27]:
single_runs = []
for _ in range(NUM_REPLICATIONS):
    handler, metrics = await run_and_measure(
        single_agent,
        single_agent_llm,
        single_agent_instruction,
        single_agent_max_steps,
    )
    single_runs.append((handler, metrics))
    print(metrics)

{'task_success': False, 'wall_seconds': 8, 'steps': 4, 'rollout_chars': 5224, 'prompt_chars': 53330}
{'task_success': True, 'wall_seconds': 323, 'steps': 87, 'rollout_chars': 158544, 'prompt_chars': 19460658}
{'task_success': True, 'wall_seconds': 752, 'steps': 180, 'rollout_chars': 331757, 'prompt_chars': 83537873}


In [28]:
coordinator_runs = []
for _ in range(NUM_REPLICATIONS):
    handler, metrics = await run_and_measure(
        coordinator,
        subagents_llm,
        coordinator_instruction,
        coordinator_max_steps,
    )
    coordinator_runs.append((handler, metrics))
    print(metrics)

{'task_success': True, 'wall_seconds': 240, 'steps': 6, 'rollout_chars': 23171, 'prompt_chars': 3750841}
{'task_success': True, 'wall_seconds': 178, 'steps': 6, 'rollout_chars': 26144, 'prompt_chars': 2333845}
{'task_success': True, 'wall_seconds': 205, 'steps': 5, 'rollout_chars': 22868, 'prompt_chars': 2570868}


### (Listing 13) Trajectory evaluations of individual runs

In [29]:
single_evals = []
for handler, _ in single_runs:
    single_evals.append(await judge(handler))

coordinator_evals = []
for handler, _ in coordinator_runs:
    coordinator_evals.append(await judge(handler))

### Evaluation Summary

In [30]:
import pandas as pd

from llm_agents_from_scratch.notebook_utils import set_dataframe_display_options

# sets display options for pd.DataFrame in notebooks
set_dataframe_display_options()

In [31]:
def summarize(
    runs: list[tuple[LLMAgent.TaskHandler, dict[str, Any]]],
    evals: list[TrajectoryEvalRubric],
) -> pd.DataFrame:
    """Shape a system's runs and evaluations into a pd.DataFrame."""
    df = pd.DataFrame(
        [
            {**metrics, **e.model_dump(exclude={"summary"})}
            for (_, metrics), e in zip(runs, evals, strict=True)
        ],
    )
    return pd.concat(
        [df, pd.DataFrame([df.sum(), df.mean()], index=["TOTAL", "AVG"])],
    )


single_df = summarize(single_runs, single_evals)
single_df

,task_success,wall_seconds,steps,rollout_chars,prompt_chars,no_fabricated_results,followed_doubling_schedule,followed_output_format
0,False,8.0,4.000000,5224.0,5.333000e+04,True,True,False
1,True,323.0,87.000000,158544.0,1.946066e+07,True,True,True
2,True,752.0,180.000000,331757.0,8.353787e+07,True,True,True
TOTAL,2.0,1083.0,271.000000,495525.0,1.030519e+08,3.0,3.0,2.0
AVG,0.666667,361.0,90.333333,165175.0,3.435062e+07,1.0,1.0,0.666667


In [32]:
coordinator_df = summarize(coordinator_runs, coordinator_evals)
coordinator_df

,task_success,wall_seconds,steps,rollout_chars,prompt_chars,no_fabricated_results,followed_doubling_schedule,followed_output_format
0,True,240.000000,6.000000,23171.0,3.750841e+06,True,True,True
1,True,178.000000,6.000000,26144.0,2.333845e+06,True,True,True
2,True,205.000000,5.000000,22868.0,2.570868e+06,True,True,True
TOTAL,3.0,623.000000,17.000000,72183.0,8.655554e+06,3.0,3.0,3.0
AVG,1.0,207.666667,5.666667,24061.0,2.885185e+06,1.0,1.0,1.0


In [33]:
summary_df = pd.DataFrame(
    {
        "single agent": [e.summary for e in single_evals],
        "coordinator + subagents": [e.summary for e in coordinator_evals],
    },
)
summary_df

,single agent,coordinator + subagents
0,"The agent correctly followed the doubling schedule and used only real tool results, but failed to complete the task, producing no final JSON output due to a malformed tool-call decision.","The coordinator dispatched exactly 20 successful pi-estimator subagents, collected their sample_ids, computed the standard error, and returned only the required JSON, with subagent results reporting proper doubling of sample points."
1,"The agent correctly generated 20 independent samples, doubling points as needed until each estimate fell in range, and called compute_standard_error, producing the exact JSON final output.","The coordinator dispatched all 20 pi-estimator subagents in batches of at most 5, collected the real sample_ids, and correctly called compute_standard_error with them, producing the final JSON output; subagent results show doubling of sample sizes as required."
2,"The agent independently ran all 20 samples, correctly doubling points (1M, 2M, 4M, ...) until each estimate fell in the target range, then computed the standard error with real tool results and returned the exact requested JSON.","The coordinator dispatched 20 pi-estimator subagents in batches of 5, collected all sample_ids from actual subagent results, and correctly called compute_standard_error to obtain the reported standard error; subagent logs confirm doubling-based sample growth and the final response was the exact JSON structure."


## (Listing 14) Comparing the two systems

The `AVG` rows side by side. The last column divides the subagent
system's value by the single agent's: below 1 means the subagent system
used less.

In [34]:
comparison_df = pd.DataFrame(
    {
        "single agent": single_df.loc["AVG"],
        "coordinator + subagents": coordinator_df.loc["AVG"],
    },
)
comparison_df["ratio"] = (
    comparison_df["coordinator + subagents"] / comparison_df["single agent"]
)
comparison_df

,single agent,coordinator + subagents,ratio
task_success,0.666667,1.0,1.5
wall_seconds,361.0,207.666667,0.575254
steps,90.333333,5.666667,0.062731
rollout_chars,165175.0,24061.0,0.14567
prompt_chars,34350620.333333,2885184.666667,0.083992
no_fabricated_results,1.0,1.0,1.0
followed_doubling_schedule,1.0,1.0,1.0
followed_output_format,0.666667,1.0,1.5


On a per-token API, the `prompt_chars` ratio is roughly the ratio of the
two systems' input bills. On a flat plan like Ollama Cloud, it is how
much of the plan's usage limits each system consumes.

In [35]:
# write results to json
single_df.to_json("single_agent_df.json")
coordinator_df.to_json("coordinator_df.json")
summary_df.to_json("summary_df.json")